In [ ]:
'''libraries'''
#Data
import pandas as pd

#Plots
import matplotlib.pyplot as plt

#math
import numpy as np

#Constants
from scipy.constants import physical_constants
m_u=physical_constants['atomic mass constant energy equivalent in MeV'][0]
from scipy.constants import speed_of_light as c

#Usefull
from tqdm.notebook import tqdm
import os
from scipy.interpolate import interp1d
import pynucastro as pyna
#%matplotlib widget

Read Data from the Snaps


In [ ]:
input_rates='Nuclear_Data\run1\Reaclib_18_9_20'
input_masses='Nuclear_Data\run1\Reaclib_18_9_20'
folder = 'Runs/run1'

files = os.listdir(folder+'\snaps')

abundances_winnet=[]
time_Winnet=[]


for file in tqdm(files):
    time_i=float(np.loadtxt(os.path.join(folder, file), skiprows=1,max_rows=1, usecols=0))
    if time_i>1:
        abundances_winnet.append(np.loadtxt(os.path.join(folder, file), skiprows=3))
        time_Winnet.append(float(np.loadtxt(os.path.join(folder, file), skiprows=1,max_rows=1, usecols=0)))

abundances_winnet = np.array(abundances_winnet)
time_Winnet = np.array(time_Winnet) 


Interpolation to have a smooth Xi curve

In [ ]:
initial_time=1 #seconds
final_time=4.32E7 #seconds 500 days= 4.32E7 seconds 100 days= 8.64E7 seconds
N_steps=100000
time = np.exp(np.linspace(np.log(initial_time), np.log(final_time), N_steps))
abundances_time= np.empty((abundances_winnet.shape[1],N_steps))

for i in tqdm(range(abundances_winnet.shape[1])):
    spline_interp = interp1d(time_Winnet, abundances_winnet[:,i,3], kind='linear')
    abundances_time[i] = spline_interp(time)


def index_n_z(n,z):
    idx_start = np.searchsorted(abundances_winnet[0,:,1], z, side='left')
    idx_end = np.searchsorted(abundances_winnet[0,:,1], z, side='right')
    if idx_end>idx_start:
        j=np.searchsorted(abundances_winnet[0,:,0][idx_start:idx_end],n)+idx_start
        if abundances_winnet[0,:,0][j]==n and abundances_winnet[0,:,1][j]==z:
            return j
        else:
            return 'None'
    else:
        return 'None'
    
def Xi_time_n_z(n,z):

    j=index_n_z(n,z)
    if j=='None':
        return np.zeros(N_steps)
    elif abundances_winnet[0,:,0][j]==n and abundances_winnet[0,:,1][j]==z:
        return abundances_time[j]
    else:
        return np.zeros(N_steps)
    

Rates used in the simulation

In [ ]:
rates_Reaclib_winnet=pyna.rates.library.Library(
    libfile=input_rates
    )

# Remove duplicate links from the library

for pair in rates_Reaclib_winnet.find_duplicate_links():
    if pair[0].eval_deriv(1e9)==0:
        rates_Reaclib_winnet.remove_rate(pair[1])
    else:
        rates_Reaclib_winnet.remove_rate(pair[0])


filter_alpha=pyna.RateFilter(
    products=['he4'],
    exact=False,
    max_reactants=1,
    max_products=2,
    filter_function=lambda r: r.Q>0 and r.reactants[0].Z==r.products[0].Z+r.products[1].Z)

rates_alpha=rates_Reaclib_winnet.filter(filter_alpha)

filter_beta_minus=pyna.RateFilter(
    max_reactants=1,
    max_products=1,
    filter_function=lambda r: r.Q>0 and r.reactants[0].Z==r.products[0].Z-1)

rates_beta_minus=rates_Reaclib_winnet.filter(filter_beta_minus)

print('number of alpha decays '+str(len(rates_alpha.get_rates()))+', number of beta decays '+str(len(rates_beta_minus.get_rates())))


Calcualtion of $\epsilon (t)$

In [ ]:

e_b=np.zeros(N_steps)
e_a=np.zeros(N_steps)
e_b_nuclei=[]
e_a_nuclei=[]
nuclei_b=[]
nuclei_a=[]

y_alpha=np.zeros(len(time_Winnet))
n_alpha=np.zeros(len(time_Winnet))

for beta_rate in tqdm(rates_beta_minus.get_rates()):
    nuclei=beta_rate.reactants[0]
    decay_rate_i=beta_rate.eval(1e9)
    Q_i=beta_rate.Q
    Z_i=nuclei.Z
    N_i=nuclei.N
    m_i=nuclei.dm+(Z_i+N_i)*m_u
    X_i=Xi_time_n_z(N_i,Z_i)
    e=decay_rate_i*(Q_i*X_i*c**2)*np.exp(-decay_rate_i*time)/m_i
    e_b+=e
    e_b_nuclei.append(e)
    nuclei_b.append(nuclei)
    

for alpha_rate in tqdm(rates_alpha.get_rates()):
    nuclei=alpha_rate.reactants[0]
    Z_i=nuclei.Z
    N_i=nuclei.N
    if index_n_z(N_i,Z_i)!='None':
        decay_rate_i=alpha_rate.eval(1e9)
        Q_i=alpha_rate.Q
        m_i=nuclei.dm+(Z_i+N_i)*m_u
        X_i=Xi_time_n_z(N_i,Z_i)
        e=decay_rate_i*(Q_i*X_i*c**2)*np.exp(-decay_rate_i*time)/m_i
        e_a+=e
        e_a_nuclei.append(e)
        nuclei_a.append(nuclei)
        
        Y_i=abundances_winnet[:, index_n_z(N_i,Z_i), 2]
        y_alpha+=(decay_rate_i)*Y_i
        n_alpha+=(decay_rate_i)*Y_i*np.exp(-decay_rate_i*time_Winnet)


e_a_erg=e_a*1e4
e_b_erg=e_b*1e4
e_a_nuclei_erg=np.array(e_a_nuclei)*1e4
e_b_nuclei_erg=np.array(e_b_nuclei)*1e4
e_a_nuclei_frac=e_a_nuclei/e_a
e_b_nuclei_frac=e_b_nuclei/e_b

In [ ]:
'''find relevant nuclei'''
time_days=time/(24*60*60)
i_initial=np.searchsorted(time_days, 0.1)
i_Final=100000

i_s=np.argsort(-np.max(e_b_nuclei_frac[:,i_initial:i_Final], axis=1))
e_b_nuclei_sorted=[e_b_nuclei_erg[i] for i in i_s]
nuclei_b_sorted=[nuclei_b[i] for i in i_s]

i_s=np.argsort(-np.max(e_a_nuclei_frac[:,i_initial:i_Final], axis=1))
e_a_nuclei_sorted=[e_a_nuclei_erg[i] for i in i_s]
nuclei_a_sorted=[nuclei_a[i] for i in i_s]

In [ ]:
#termalization

#important parameters
Mey=0.05
Vey=0.15

tb=12.9*((Mey/0.01)**(2/3))*((Vey/0.2)**(-2))*24*60*60 #termalization beta particles
ty=0.3*np.sqrt(Mey/0.01)*(0.2/Vey)*24*60*60 ##termalization gamma particles
f_gamma=1-np.exp(-(ty/time)**2)
f_electrons=(1+time/tb)**(-1)
f_beta=0.2*f_electrons+0.45*f_gamma
f_alpha=(1+time/(3*tb))**(-1)

e_b_ef=e_b_erg*f_beta
e_a_ef=e_a_erg*f_alpha


In [ ]:
'''Preliminar figures'''
plt.plot(time_days,e_a_ef,label=r'alpha ',linestyle=":")
plt.plot(time_days,e_b_ef,label=r'beta ')
plt.plot(time_days,e_a_ef+e_b_ef,label=r'total ' )
for i in range(5):
    plt.plot(time_days,e_a_nuclei_sorted[i]*f_alpha,label=f'alpha {nuclei_a_sorted[i]}',linestyle=":")
    plt.plot(time_days,e_b_nuclei_sorted[i]*f_beta,label=f'beta {nuclei_b_sorted[i]}')
plt.yscale('log')
plt.xscale('log')
plt.xlabel('Time[Days]')
plt.ylabel('epsilon[erg/g*s]')
plt.legend()
plt.ylim(1e2,1e16)
plt.xlim(0.0001,500)


In [ ]:
'''
#save results
pd.DataFrame({'Epsilon_alpha':e_a_ef,'Time [Days]':time_days}).to_csv('Results/'+folder+'/alpha.csv',index=False)
pd.DataFrame({'Epsilon_beta':e_b_ef,'Time [Days]':time_days}).to_csv('Results/'+folder+'/beta.csv',index=False)
for i in range(20):
    pd.DataFrame({'time_days':time_days,f'{nuclei_b_sorted[i]} top {i+1}':e_b_nuclei_sorted[i]}).to_csv(f'Results/'+folder+'/b_top/e_r_{i}.csv',index=False)
for i in range(20):
    pd.DataFrame({'time_days':time_days,f'{nuclei_a_sorted[i]} top {i+1}':e_a_nuclei_sorted[i]}).to_csv(f'Results/'+folder+'/a_top/e_r_{i}.csv',index=False)
'''